using pytorch

In [2]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    accuracy_score, roc_auc_score, classification_report
)
from sklearn.preprocessing import label_binarize

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

In [3]:
# ── Config ───────────────────────────────────────────────────────────────────
folder_path = r"D:\GSoC\DeepLense\dataset"
TYPES       = ['no', 'sphere', 'vort']
LABEL_MAP   = {'no': 0, 'sphere': 1, 'vort': 2}
NUM_CLASSES = 3
WIDTH       = 128
NUM_EPOCHS  = 30
BATCH_SIZE  = 128
LR          = 1e-4
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cpu


In [4]:
# ── 1. Data loading ───────────────────────────────────────────────────────────
def load_data(split):
    """
    Loads .npy files from  <folder_path>/<split>/<class>/
    Each file is assumed to be shape (C, H, W) — kept as-is for PyTorch.
    Returns arrays of shape (N, C, H, W) and (N,) integer labels.
    """
    data, labels = [], []
    for t in TYPES:
        path  = os.path.join(folder_path, split, t)
        label = LABEL_MAP[t]
        for npy_name in os.listdir(path):
            arr = np.load(os.path.join(path, npy_name))   # (C, H, W)
            data.append(arr)
            labels.append(label)
    return np.array(data, dtype=np.float32), np.array(labels, dtype=np.int64)


X_train, Y_train = load_data('train')
X_val,   Y_val   = load_data('val')

# Original Keras code did .transpose(0,2,3,1) to go (N,C,H,W)→(N,H,W,C).
# PyTorch wants (N,C,H,W) natively, so NO transpose needed here.
print(f"Train shape: {X_train.shape}, Labels: {Y_train.shape}")
print(f"Val   shape: {X_val.shape},   Labels: {Y_val.shape}")

Train shape: (30000, 1, 150, 150), Labels: (30000,)
Val   shape: (7500, 1, 150, 150),   Labels: (7500,)


In [9]:
# ── 2. Dataset & DataLoader ───────────────────────────────────────────────────
class LensDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.tensor(images)          # (N, C, H, W)  float32
        self.labels = torch.tensor(labels)          # (N,)           int64

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


train_loader = DataLoader(LensDataset(X_train, Y_train),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(LensDataset(X_val, Y_val),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

In [10]:
# ── 3. Model ─────────────────────────────────────────────────────────────────
def get_encoder(width=WIDTH, in_channels=1):
    """
    ResNet-50 backbone (no pretrained weights, matching weights=None)
    with the first conv adapted to accept `in_channels` input channels.
    Followed by the same Dense→BN→Dropout head as the Keras version.
    """
    resnet = models.resnet50(weights=None)

    # Patch conv1 for single-channel (or any non-3-channel) input
    resnet.conv1 = nn.Conv2d(
        in_channels, 64,
        kernel_size=7, stride=2, padding=3, bias=False
    )
    in_feat   = resnet.fc.in_features          # 2048
    resnet.fc = nn.Identity()                  # drop default FC; GAP is already inside ResNet

    head = nn.Sequential(
        # 1st dense block
        nn.Linear(in_feat,    width * 8),      # 1024
        nn.ReLU(),
        nn.BatchNorm1d(width * 8),
        nn.Dropout(0.5),
        # 2nd dense block
        nn.Linear(width * 8,  width * 4),      # 512
        nn.ReLU(),
        nn.BatchNorm1d(width * 4),
        nn.Dropout(0.5),
        # 3rd dense block
        nn.Linear(width * 4,  width),          # 128
        nn.ReLU(),
        nn.BatchNorm1d(width),
        nn.Dropout(0.3),
    )
    return nn.Sequential(resnet, head)


class SupervisedModel(nn.Module):
    def __init__(self, width=WIDTH, num_classes=NUM_CLASSES, in_channels=1):
        super().__init__()
        self.encoder    = get_encoder(width, in_channels)
        self.classifier = nn.Linear(width, num_classes)

    def forward(self, x):
        return self.classifier(self.encoder(x))    # raw logits


# Infer in_channels from data
in_channels = X_train.shape[1]
model       = SupervisedModel(in_channels=in_channels).to(DEVICE)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
    model = nn.DataParallel(model)

print(model)

SupervisedModel(
  (encoder): Sequential(
    (0): ResNet(
      (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inp

In [11]:
# ── 4. Optimizer & loss ───────────────────────────────────────────────────────
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()     # raw logits + integer labels (no one-hot needed)

In [12]:
# ── 5. Training loop ──────────────────────────────────────────────────────────
def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, lbls in tqdm(loader, leave=False):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)

            if training:
                optimizer.zero_grad()

            logits = model(imgs)
            loss   = criterion(logits, lbls)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            correct    += (logits.argmax(dim=1) == lbls).sum().item()
            total      += imgs.size(0)

            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(lbls.cpu().numpy())

    avg_loss = total_loss / total
    acc      = correct / total

    # Multi-class AUC (one-vs-rest, macro average)
    y_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
    auc_score = roc_auc_score(y_bin, np.array(all_probs),
                              multi_class='ovr', average='macro')
    return avg_loss, acc, auc_score


history = {k: [] for k in ['loss', 'acc', 'auc', 'val_loss', 'val_acc', 'val_auc']}

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, tr_auc = run_epoch(train_loader, training=True)
    va_loss, va_acc, va_auc = run_epoch(val_loader,   training=False)

    for k, v in zip(history, [tr_loss, tr_acc, tr_auc, va_loss, va_acc, va_auc]):
        history[k].append(v)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"loss {tr_loss:.4f}  acc {tr_acc:.4f}  auc {tr_auc:.4f} | "
          f"val_loss {va_loss:.4f}  val_acc {va_acc:.4f}  val_auc {va_auc:.4f}")

  0%|          | 0/235 [00:00<?, ?it/s]d:\GSoC\DeepLense\DeepLense\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 0/59 [00:00<?, ?it/s]           d:\GSoC\DeepLense\DeepLense\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 01/30 | loss 1.1859  acc 0.3328  auc 0.5004 | val_loss 1.1161  val_acc 0.3344  val_auc 0.5061


  0%|          | 0/235 [00:00<?, ?it/s]d:\GSoC\DeepLense\DeepLense\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 0/59 [00:00<?, ?it/s]           d:\GSoC\DeepLense\DeepLense\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 02/30 | loss 1.1526  acc 0.3362  auc 0.5039 | val_loss 1.1038  val_acc 0.3355  val_auc 0.5001


  0%|          | 0/235 [00:00<?, ?it/s]d:\GSoC\DeepLense\DeepLense\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 0/59 [00:00<?, ?it/s]           d:\GSoC\DeepLense\DeepLense\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Epoch 07/30 | loss 1.1143  acc 0.3454  auc 0.5143 | val_loss 1.0998  val_acc 0.3419  val_auc 0.5128


  0%|          | 0/235 [00:00<?, ?it/s]d:\GSoC\DeepLense\DeepLense\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 0/59 [00:00<?, ?it/s]            d:\GSoC\DeepLense\DeepLense\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 08/30 | loss 1.1131  acc 0.3481  auc 0.5102 | val_loss 1.0998  val_acc 0.3445  val_auc 0.5137


  0%|          | 0/235 [00:00<?, ?it/s]d:\GSoC\DeepLense\DeepLense\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
# ── 6. Save ───────────────────────────────────────────────────────────────────
torch.save(model.state_dict(), 'supervised_resnet_baseline.pth')
with open('supervised_baseline_resnet50.pkl', 'wb') as f:
    pickle.dump(history, f)
print("Model saved to supervised_resnet_baseline.pth")

In [ ]:
# ── 7. Reload & evaluate ──────────────────────────────────────────────────────
model_reloaded = SupervisedModel(in_channels=in_channels).to(DEVICE)
model_reloaded.load_state_dict(
    torch.load('supervised_resnet_baseline.pth', map_location=DEVICE)
)
model_reloaded.eval()

all_preds, all_probs_eval, all_true = [], [], []
with torch.no_grad():
    for imgs, lbls in val_loader:
        logits = model_reloaded(imgs.to(DEVICE))
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_probs_eval.extend(probs)
        all_true.extend(lbls.numpy())

all_true       = np.array(all_true)
all_preds      = np.array(all_preds)
all_probs_eval = np.array(all_probs_eval)

print("\n── Classification Report ──")
print(classification_report(all_true, all_preds,
                             target_names=TYPES))

print("Confusion Matrix:")
print(confusion_matrix(all_true, all_preds))

y_bin     = label_binarize(all_true, classes=[0, 1, 2])
final_auc = roc_auc_score(y_bin, all_probs_eval, multi_class='ovr', average='macro')
print(f"Macro AUC: {final_auc:.4f}")